# FedMed-FL — Google Colab Runner

**Privacy-Preserving Collaborative Medical Diagnostics Using Decentralized Federated Learning**

Runs the whole capstone experiment end to end. Runtime: **CPU is fine** — the model is 14,280 parameters.

`Runtime -> Run all`, then read the outputs. Total time ~5 minutes.

> The one thing this notebook **cannot** do is the real two-laptop Flower deployment — Colab cannot open a gRPC socket between two machines. Do that part locally in VS Code (README Section 6).

## 1. Setup

Upload the `FedMed-FL` folder to your Google Drive first, then point `PROJECT` at it. Or unzip it straight into the Colab VM with the second cell.

In [ ]:
# Option A — project lives in Google Drive (persists between sessions)
from google.colab import drive
drive.mount('/content/drive')
PROJECT = '/content/drive/MyDrive/FedMed-FL'   # <-- edit this path


In [ ]:
# Option B — upload the zip instead (faster, but lost when the runtime dies)
# from google.colab import files
# up = files.upload()            # choose FedMed-FL.zip
# !unzip -q -o FedMed-FL.zip -d /content
# PROJECT = '/content/FedMed-FL'


In [ ]:
import os, sys
os.chdir(PROJECT)
sys.path.insert(0, PROJECT)
print('working in:', os.getcwd())
print(os.listdir('.'))


In [ ]:
# Colab already has torch, numpy, pandas, sklearn, matplotlib.
# Only Flower is missing, and it is needed solely for federated/*.py.
!pip install -q 'flwr>=1.16,<2.0'
import torch; print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available())


## 2. Look at the data before training anything

The non-IID skew is not a simulation trick — it falls out of geography. Watch Srinagar's Dengue and Chikungunya counts.

In [ ]:
import config as cfg
from src.data_generator import build_all
from src import validation, data_utils

raw = build_all()
for cid in cfg.CLIENT_IDS:
    print(f'{cid:<22} {len(raw[cid]):>5} raw admissions')
raw['H1_Max_Delhi'].head()


In [ ]:
# Phase-1 quarantine + Phase-2 quality gate, per hospital
verified = {}
for cid in cfg.CLIENT_IDS:
    clean, _ = validation.run_validation_pipeline(raw[cid], cid)
    verified[cid] = data_utils.encode(clean)[1]


In [ ]:
table = data_utils.label_distribution_table(verified)
display(table)

import matplotlib.pyplot as plt
table.drop(columns=['TOTAL']).plot(kind='bar', stacked=True, figsize=(11,4.5), colormap='tab20')
plt.title('Non-IID label distribution across hospital clients')
plt.ylabel('Verified training cases'); plt.legend(bbox_to_anchor=(1.02,1), fontsize=8)
plt.tight_layout(); plt.show()


## 3. Main experiment — federated + record-level DP

This is the number you report. ~2 minutes on CPU.

In [ ]:
!python src/run_simulation.py


## 4. Centralized baseline — the privacy-violating upper bound

What you *could* achieve if HIPAA, GDPR and DISHA did not exist. The gap is the price of privacy — report it, do not hide it.

In [ ]:
!python src/run_centralized.py


## 5. Ablations

Each of these is a row in your results section.

In [ ]:
# 5a. Utility ceiling: federation with NO differential privacy
!python src/run_simulation.py --no-dp --rounds 20


In [ ]:
# 5b. Client-level DP collapses at K=3 hospitals.
#     This is a DELIBERATE negative result — see README Section 5.
!python src/run_simulation.py --dp-mode client --sigma 1.0 --rounds 15


In [ ]:
# 5c. What is the Phase-2 validation layer actually worth?
!python src/run_simulation.py --no-validation --rounds 20
!python src/run_simulation.py --no-loss-filter --rounds 20


In [ ]:
# 5d. Privacy-utility trade-off curve (slowest cell, ~4 min)
!python src/run_simulation.py --privacy-sweep


## 6. Non-IID sweep over the Dirichlet alpha

Answers 'how does your method behave as heterogeneity increases?' — the question a reviewer always asks.

In [ ]:
import numpy as np
from src.data_utils import dirichlet_partition

y = np.concatenate([verified[c] for c in cfg.CLIENT_IDS])
for alpha in [0.1, 0.5, 1.0, 100.0]:
    parts = dirichlet_partition(y, num_clients=3, alpha=alpha)
    dist = [np.bincount(y[p], minlength=cfg.NUM_CLASSES) for p in parts]
    print(f'alpha={alpha:<7} (lower = more skewed)')
    for i, d in enumerate(dist):
        print(f'   client {i}: {d.tolist()}')


## 7. Clinical Decision Support demo

Cases 2 and 3 are byte-identical presentations that differ only in region and month. Case 5 is the counter-example you should raise before your examiner does.

In [ ]:
!python src/cdss.py


In [ ]:
# Try your own patient
from src.cdss import load_model, predict
model = load_model()

patient = {
    'age': 34, 'gender': 1, 'region': 'Tropical_Monsoon', 'month': 9,
    'fever_days': 7, 'temp_max_c': 39.4,
    'joint_pain': 1, 'headache': 1, 'chills_rigor': 1, 'vomiting': 1,
    'platelet_count_k': 120, 'wbc_count_k': 4.8,
    'hemoglobin_gdl': 10.8, 'crp_mgl': 70, 'spo2_pct': 96,
}
predict(model, patient, top_k=8)


## 8. Collect every figure for the manuscript

In [ ]:
import glob
from IPython.display import Image, display

for f in sorted(glob.glob('outputs/*.png')):
    print(f); display(Image(f, width=760))


In [ ]:
# Zip the outputs and download them
!zip -qr fedmed_outputs.zip outputs
from google.colab import files
files.download('fedmed_outputs.zip')


## 9. What Colab cannot do

The real two-laptop Flower deployment (`federated/server.py` + `federated/client.py`) needs two machines on the same network. Run that locally in VS Code — see README Section 6.

You can still exercise the Flower client logic here, without the network:

In [ ]:
import numpy as np
from federated.client import HospitalClient
from federated.server import build_test_loader
from src.model import build_model, set_weights
from src import engine, data_utils, aggregate

test_loader = build_test_loader()
gm = build_model(data_utils.NUM_FEATURES, cfg.SEED)
params = [v.cpu().numpy() for v in gm.state_dict().values()]
clients = [HospitalClient(c) for c in cfg.CLIENT_IDS]

for rnd in range(1, 6):
    conf = {'server_round': rnd, 'local_epochs': 3, 'dp_mode': 'record',
            'sigma': 1.5, 'clip_norm': 1.0}
    ws, ns = [], []
    for c in clients:
        w, n, _ = c.fit(params, conf)
        ws.append([np.asarray(x) for x in w]); ns.append(n)
    params = aggregate.fedavg(ws, ns)
    set_weights(gm, params)
    ev = engine.evaluate(gm, test_loader)
    print(f"ROUND {rnd}: acc={ev['accuracy']:.4f}  macro-F1={ev['macro_f1']:.4f}")
